In [4]:
# Setup
import duckdb
from pathlib import Path

# Resolve the database path relative to the ingestion/ folder
DB_PATH = str(Path("../raw_data/my_cost_of_living.duckdb").resolve())
print(DB_PATH)

C:\Projects\my-cost-of-living\raw_data\my_cost_of_living.duckdb


In [5]:
# Load source files into the raw schema
sources = {
    "pricecatcher_2026_10": "https://storage.data.gov.my/pricecatcher/pricecatcher_2026-10.parquet",
    "lookup_premise": "https://storage.data.gov.my/pricecatcher/lookup_premise.parquet",
    "lookup_item": "https://storage.data.gov.my/pricecatcher/lookup_item.parquet",
    "fuelprice": "https://storage.data.gov.my/commodities/fuelprice.parquet",
}

con = duckdb.connect(DB_PATH)
con.execute("create schema if not exists raw")

for table_name, url in sources.items():
    # Idempotent load with lineage metadata: load time and source file
    con.execute(
        f"""
        create or replace table raw.{table_name} as
        select
            *,
            current_timestamp as _loaded_at,
            '{url}' as _source_url
        from read_parquet('{url}')
        """
    )
    row_count = con.execute(f"select count(*) from raw.{table_name}").fetchone()[0]
    print(f"{table_name}: {row_count} rows")

#
con.close()

pricecatcher_2026_10: 1359706 rows
lookup_premise: 3932 rows
lookup_item: 797 rows
fuelprice: 971 rows


In [6]:
# Inspect loaded tables (read-only)
con = duckdb.connect(DB_PATH, read_only=True)
tables = con.sql("show all tables").df()
con.close()
tables

,database,schema,name,column_names,column_types,temporary
0,my_cost_of_living,main,stg_fuelprice,"[series_type, date, ron95, ron97, diesel, dies...","[VARCHAR, DATE, DOUBLE, DOUBLE, DOUBLE, DOUBLE...",False
1,my_cost_of_living,raw,fuelprice,"[series_type, date, ron95, ron97, diesel, dies...","[VARCHAR, DATE, DOUBLE, DOUBLE, DOUBLE, DOUBLE...",False
2,my_cost_of_living,raw,lookup_item,"[item_code, item, unit, item_group, item_categ...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, T...",False
3,my_cost_of_living,raw,lookup_premise,"[premise_code, premise, address, premise_type,...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, VARCHAR, V...",False
4,my_cost_of_living,raw,pricecatcher_2026_10,"[date, premise_code, item_code, price, _loaded...","[DATE, BIGINT, BIGINT, DOUBLE, TIMESTAMP WITH ...",False


In [19]:
# Quick peek (read-only)
con = duckdb.connect(DB_PATH, read_only=True)
df = con.sql("select * from raw.pricecatcher_2026_10 limit 10").df()
con.close()
df

,date,premise_code,item_code,price,_loaded_at,_source_url
0,2026-10-10,2,1,11.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
1,2026-10-10,2,47,9.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
2,2026-10-10,2,55,9.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
3,2026-10-10,2,70,18.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
4,2026-10-10,2,88,10.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
5,2026-10-10,2,92,10.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
6,2026-10-10,2,93,20.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
7,2026-10-10,2,94,15.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
8,2026-10-10,2,95,12.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...
9,2026-10-10,2,96,8.0,2026-11-06 17:24:34.584225+08:00,https://storage.data.gov.my/pricecatcher/price...


In [18]:
def peek(query):
    # Run a read-only query and release the file straight away
    con = duckdb.connect(DB_PATH, read_only=True)
    df = con.sql(query).df()
    con.close()
    return df


peek("select count(*) filter (where premise_code = -1) as unknown_premise, count(*) filter (where item_code = -1) as unknown_item from raw.pricecatcher_2026_10")

,unknown_premise,unknown_item
0,0,0
